# 🔮 FORESIGHT — 07: Inventory Optimization & Working Capital Intelligence

This notebook demonstrates enterprise inventory control logic:
- **Safety Stock Modeling** (Demand, Lead Time, Combined, Quantile)
- **Service Level Tradeoff Curves** (90% to 99%)
- **Reorder Point (ROP) & Continuous Review**
- **Economic Order Quantity (EOQ)** and Total Cost Curves
- **Portfolio Working Capital Optimization**

In [ ]:
import pandas as pd
import numpy as np
from foresight.inventory.safety_stock import calculate_safety_stock, calculate_z_score
from foresight.inventory.eoq import calculate_eoq, calculate_inventory_cost_breakdown
from foresight.inventory.optimizer import InventoryOptimizer
from foresight.inventory.schema import InventoryParameters, SafetyStockMethod

# 1. Service Level vs Safety Stock Sensitivity
service_levels = [0.85, 0.90, 0.95, 0.98, 0.99, 0.999]
print("=== SERVICE LEVEL VS SAFETY STOCK SENSITIVITY ===")
print(f"{'Service Level':<15} | {'Z-Score':<10} | {'Safety Stock (Units)':<20}")
print("-" * 50)

for sl in service_levels:
    z = calculate_z_score(sl)
    ss = calculate_safety_stock(
        method=SafetyStockMethod.COMBINED_UNCERTAINTY,
        mean_demand=25.0,
        std_demand=5.0,
        mean_lead_time=7.0,
        std_lead_time=1.0,
        service_level=sl,
    )
    print(f"{sl*100:<14.1f}% | {z:<10.4f} | {ss:<20.2f}")

## 2. Wilson EOQ Total Cost Curve Analysis
Analyze the U-shaped inventory carrying cost curve balancing ordering and holding expenditures.

In [ ]:
annual_d = 10000.0
order_cost = 50.0
unit_c = 25.0
holding_r = 0.20

opt_eoq = calculate_eoq(annual_d, order_cost, unit_c, holding_r)
opt_costs = calculate_inventory_cost_breakdown(annual_d, opt_eoq, order_cost, unit_c, holding_r)

print(f"Optimal EOQ: {opt_eoq:.1f} units")
print(f"Annual Ordering Cost: ${opt_costs['annual_ordering_cost']:,.2f}")
print(f"Annual Holding Cost:  ${opt_costs['annual_holding_cost']:,.2f}")
print(f"Total Inventory Cost: ${opt_costs['total_annual_cost']:,.2f}")

## 3. SKU Optimization Execution

In [ ]:
params = InventoryParameters(
    sku_id="SKU-1001",
    store_id="STORE-001",
    current_on_hand=35.0,
    units_on_order=0.0,
    backorders=0.0,
    unit_cost=45.0,
    unit_price=80.0,
    lead_time_days=7.0,
    lead_time_std_days=1.0,
    holding_cost_annual_rate=0.20,
    fixed_order_cost=50.0,
    min_order_qty=20.0,
    target_service_level=0.95,
    forecast_daily_demand_mean=18.0,
    forecast_daily_demand_std=4.0,
)

optimizer = InventoryOptimizer()
res = optimizer.optimize_sku(params)
print("Optimization Result:")
print(res.model_dump_json(indent=2))